# Notebook Overview — Prepare Video Evidence

## Purpose

This notebook prepares lightweight video evidence metadata for the NExT-QA VideoQA project. Instead of duplicating video content, the notebook generates evidence records that identify meaningful regions of the original videos using video identifiers, timestamps, segment boundaries, and related metadata.

The resulting evidence metadata serves as the bridge between the prepared NExT-QA videos from Notebook 01 and the video knowledge base constructed in Notebook 04.

## Inputs

* Prepared NExT-QA video files from Notebook 01
* NExT-QA question-answer files
  * train.csv
  * val.csv
  * test.csv
* NExT-QA metadata resources
* Project configuration settings
* Shared video and dataset utility functions

## Outputs

* Evidence metadata CSV file
* Video inventory summary
* Evidence metadata summary report
* Sample evidence records for verification

## Processing Workflow

1. Load project configuration and dataset resources.
2. Load NExT-QA metadata and video inventory information.
3. Define the evidence metadata schema and segmentation parameters.
4. Inspect representative videos and extract video properties.
5. Generate evidence metadata records for each processed video.
6. Validate evidence metadata completeness and consistency.
7. Save evidence metadata and summary files.


### 🔷 Step 1 — Clone Required Repository Files

* Clone the project repository using sparse checkout to minimize download size and runtime initialization overhead.
* Authenticate access to the private GitHub repository using a fine-grained access token stored in Google Colab Secrets.
* Configure the local notebook workspace and change to the repository working directory.
* Verify that required repository files and directories are available for subsequent notebook execution.
* Optionally display repository paths, directory contents, and cloned files when `VERBOSE=True`.


In [ ]:
# ============================================================
# Step 1: Clone Required Repository Files
# ============================================================

VERBOSE = True

import os
from google.colab import userdata

REPO_NAME = "iterative-video-rag"
REPO_OWNER = "pgailinas"
REPO_BASE_DIR = "/content"
REPO_DIR = os.path.join(REPO_BASE_DIR, REPO_NAME)

# ------------------------------------------------------------
# Retrieve GitHub Token from Colab Secrets
# ------------------------------------------------------------

github_token = userdata.get("GITHUB_TOKEN")

if github_token is None:
    raise ValueError(
        "GITHUB_TOKEN not found in Colab Secrets."
    )

repo_url = (
    f"https://{github_token}"
    f"@github.com/{REPO_OWNER}/{REPO_NAME}.git"
)

# ------------------------------------------------------------
# Move to Base Directory
# ------------------------------------------------------------

%cd {REPO_BASE_DIR}

# ------------------------------------------------------------
# Clone Repository if Needed
# ------------------------------------------------------------

if not os.path.exists(REPO_DIR):

    if VERBOSE:
        print("Cloning required repository directories...")

    !git clone --quiet --filter=blob:none --no-checkout {repo_url}

    %cd {REPO_DIR}

    !git sparse-checkout init --cone

    !git sparse-checkout set \
        src \
        datasets \
        outputs

    !git checkout --quiet main

else:

    if VERBOSE:
        print(f"Repository already exists: {REPO_DIR}")

    %cd {REPO_DIR}

# ------------------------------------------------------------
# Verify Repository Setup
# ------------------------------------------------------------

required_paths = [
    "src",
    "datasets",
    "outputs",
    "datasets/NExT-QA",
    "datasets/NExT-QA/questions",
    "datasets/NExT-QA/metadata",
    "src/iterative_rag_config.py",
    "src/nextqa_video_cache.py",
    "src/nextqa_metadata.py",
    "src/video_evidence.py",
    "src/evidence_validation.py",
    "src/evidence_io.py",
]

for path in required_paths:

    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Required path not found: {path}"
        )

print("Repository setup complete.")

if VERBOSE:
    print(f"\nCurrent directory: {os.getcwd()}")
    print("\nRepository directories:")
    !find src datasets outputs \
        -maxdepth 2 \
        -type d \
        ! -path "*/__pycache__*" | sort



### 🔷 Step 2 — Import Libraries and Load Configuration

* Import the Python libraries required for evidence metadata generation and validation.
* Load centralized project configuration settings and constants from `iterative_rag_config.py`.
* Import reusable video utility functions from `nextqa_video_cache.py`.
* Initialize shared configuration values, paths, and runtime settings used throughout the notebook.
* Verify that required modules and configuration resources are available before continuing.


In [ ]:
# ============================================================
# Step 2: Import Libraries and Load Configuration
# ============================================================

# ------------------------------------------------------------
# Standard Library Imports
# ------------------------------------------------------------

from pathlib import Path

# ------------------------------------------------------------
# Third-Party Library Imports
# ------------------------------------------------------------

import pandas as pd

# ------------------------------------------------------------
# Project Configuration
# ------------------------------------------------------------

from src.iterative_rag_config import *

# ------------------------------------------------------------
# Reusable Project Modules
# ------------------------------------------------------------

from src.nextqa_video_cache import *
from src.nextqa_metadata import *
from src.video_evidence import *
from src.evidence_validation import *
from src.evidence_io import *

# ------------------------------------------------------------
# Import Verification
# ------------------------------------------------------------

print("Project configuration loaded successfully.")
print("Project utility modules loaded successfully.")

if VERBOSE:
    print("\nLoaded Modules:")
    print("  ✓ iterative_rag_config")
    print("  ✓ nextqa_video_cache")
    print("  ✓ nextqa_metadata")
    print("  ✓ video_evidence")
    print("  ✓ evidence_validation")
    print("  ✓ evidence_io")



### 🔷 Step 3 — Define Input and Output Paths

* Define the input directories containing NExT-QA videos, questions, and metadata resources.
* Define the output directories used to store evidence metadata and validation reports.
* Construct notebook paths using centralized project configuration values.
* Create required output directories when they do not already exist.
* Verify that required input paths are available before continuing.


In [ ]:
# ============================================================
# Step 3: Define Input and Output Paths
# ============================================================

# ------------------------------------------------------------
# NExT-QA Input Directories
# ------------------------------------------------------------

INPUT_QUESTIONS_DIR = QUESTIONS_DIR
INPUT_METADATA_DIR = METADATA_DIR
INPUT_VIDEOS_DIR = VIDEOS_DIR

# ------------------------------------------------------------
# Evidence Output Directories
# ------------------------------------------------------------

EVIDENCE_METADATA_DIR = EVIDENCE_DIR / "metadata"
EVIDENCE_REPORTS_DIR = EVIDENCE_DIR / "reports"

EVIDENCE_METADATA_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

EVIDENCE_REPORTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# ------------------------------------------------------------
# Evidence Output Files
# ------------------------------------------------------------

EVIDENCE_METADATA_CSV = (
    EVIDENCE_METADATA_DIR /
    "evidence_metadata.csv"
)

EVIDENCE_VALIDATION_CSV = (
    EVIDENCE_REPORTS_DIR /
    "evidence_validation.csv"
)

EVIDENCE_SUMMARY_CSV = (
    EVIDENCE_REPORTS_DIR /
    "evidence_summary.csv"
)

# ------------------------------------------------------------
# Verify Required Input Paths
# ------------------------------------------------------------

required_input_paths = [
    INPUT_QUESTIONS_DIR,
    INPUT_METADATA_DIR,
]

for path in required_input_paths:

    if not path.exists():

        raise FileNotFoundError(
            f"Required input path not found: {path}"
        )

print("Input and output paths initialized successfully.")

if VERBOSE:
    print("\nInput Directories")
    print("-" * 60)
    print(f"Questions : {INPUT_QUESTIONS_DIR}")
    print(f"Metadata  : {INPUT_METADATA_DIR}")
    print(f"Videos    : {INPUT_VIDEOS_DIR}")

    print("\nOutput Directories")
    print("-" * 60)
    print(f"Evidence Metadata : {EVIDENCE_METADATA_DIR}")
    print(f"Evidence Reports  : {EVIDENCE_REPORTS_DIR}")

    print("\nOutput Files")
    print("-" * 60)
    print(f"Evidence CSV : {EVIDENCE_METADATA_CSV}")
    print(f"Summary CSV  : {EVIDENCE_SUMMARY_CSV}")



### 🔷 Step 4 — Restore Local NExT-QA Video Cache

* Restore the local NExT-QA video cache using shared video cache utilities.
* Verify that extracted video files are available in the local runtime.
* Rebuild or extract video resources only when required.
* Confirm that video files are ready for evidence generation.


In [ ]:
# ============================================================
# Step 4: Restore Local NExT-QA Video Cache
# ============================================================

from google.colab import drive

# ------------------------------------------------------------
# Mount Google Drive
# ------------------------------------------------------------

GOOGLE_DRIVE_MOUNT = "/content/drive"

if not os.path.exists(GOOGLE_DRIVE_MOUNT):

    if VERBOSE:
        print("Mounting Google Drive...")

    drive.mount(GOOGLE_DRIVE_MOUNT)

else:

    if VERBOSE:
        print("Google Drive is already mounted.")

drive_root = Path(GOOGLE_DRIVE_MOUNT) / "MyDrive"

if not drive_root.exists():

    raise FileNotFoundError(
        "Unable to access Google Drive root directory."
    )

# ------------------------------------------------------------
# Configure Archive Source and Local Cache Paths
# ------------------------------------------------------------

DRIVE_DATASET_DIR = drive_root / "VideoQA_Project" / "NExT-QA"

LOCAL_ARCHIVE_DIR = DATASET_DIR / "archives"

COMBINED_ARCHIVE_PATH = (
    LOCAL_ARCHIVE_DIR /
    "NExTVideo_combined.zip"
)

# ------------------------------------------------------------
# Define Required Archive Files
# ------------------------------------------------------------

required_archive_files = [
    "NExTVideo.z01",
    "NExTVideo.z02",
    "NExTVideo.z03",
    "NExTVideo.z04",
    "NExTVideo.z05",
    "NExTVideo.z06",
    "NExTVideo.zip",
]

# ------------------------------------------------------------
# Restore Local Video Cache
# ------------------------------------------------------------

video_cache_restore_summary = restore_nextqa_video_cache(
    source_archive_dir=DRIVE_DATASET_DIR,
    local_archive_dir=LOCAL_ARCHIVE_DIR,
    local_videos_dir=INPUT_VIDEOS_DIR,
    combined_archive_path=COMBINED_ARCHIVE_PATH,
    required_archive_files=required_archive_files,
    force_rebuild=False,
    force_extract=False,
    verbose=VERBOSE,
)

print("\nLocal NExT-QA video cache is ready.")



### 🔷 Step 5 — Load NExT-QA Metadata and Video Inventory

* Load NExT-QA question-answer files and supporting dataset metadata.
* Load the video inventory and identify videos available for processing.
* Associate video identifiers with dataset splits and metadata records.
* Verify that required metadata resources contain valid records.
* Generate summary statistics for videos and question-answer datasets.



In [ ]:
# ============================================================
# Step 5: Load NExT-QA Metadata and Video Inventory
# ============================================================

# ------------------------------------------------------------
# Load NExT-QA Annotation Files
# ------------------------------------------------------------

split_annotations = load_nextqa_split_annotations(
    annotations_dir=INPUT_QUESTIONS_DIR,
    verbose=VERBOSE,
)

annotations_df = combine_nextqa_annotations(
    split_dataframes=split_annotations,
    verbose=VERBOSE,
)

# ------------------------------------------------------------
# Build Local Video Inventory
# ------------------------------------------------------------

video_inventory_df = build_nextqa_video_inventory(
    videos_dir=INPUT_VIDEOS_DIR,
    verbose=VERBOSE,
)

# ------------------------------------------------------------
# Attach Video Inventory Information
# ------------------------------------------------------------

annotations_with_videos_df = (
    attach_video_inventory_to_annotations(
        annotations=annotations_df,
        video_inventory=video_inventory_df,
        verbose=VERBOSE,
    )
)

# ------------------------------------------------------------
# Generate Split Summary
# ------------------------------------------------------------

split_summary_df = summarize_nextqa_splits(
    annotations=annotations_df,
)

# ------------------------------------------------------------
# Verify Annotation Coverage
# ------------------------------------------------------------

coverage_summary = verify_annotation_video_coverage(
    annotations=annotations_df,
    video_inventory=video_inventory_df,
    verbose=VERBOSE,
)

# ------------------------------------------------------------
# Display Summary
# ------------------------------------------------------------

print("\nNExT-QA metadata and video inventory loaded successfully.")

print(f"Annotation records : {len(annotations_df):,}")
print(f"Video inventory    : {len(video_inventory_df):,}")

if VERBOSE:
    print("\nSplit Summary")
    print("-" * 60)
    display(split_summary_df)



### 🔷 Step 6 — Define Evidence Metadata Schema

* Define the structure and fields used to represent video evidence records.
* Specify required metadata attributes, identifiers, timestamps, and evidence relationships.
* Establish data types and validation requirements for evidence records.
* Ensure the schema supports downstream retrieval, analysis, and VideoQA workflows.
* Create the evidence metadata template used throughout the notebook.



In [ ]:
# ============================================================
# Step 6: Define Evidence Metadata Schema
# ============================================================

# ------------------------------------------------------------
# Required Evidence Metadata Columns
# ------------------------------------------------------------

EVIDENCE_SCHEMA = {
    "evidence_id": "str",
    "video_id": "str",
    "split": "str",
    "video_path": "str",
    "segment_index": "int",

    "evidence_level": "int",
    "parent_evidence_id": "str",
    "segment_strategy": "str",

    "start_time_sec": "float",
    "midpoint_time_sec": "float",
    "end_time_sec": "float",
    "duration_sec": "float",

    "start_frame_idx": "int",
    "midpoint_frame_idx": "int",
    "end_frame_idx": "int",

    "fps": "float",
    "frame_count": "int",
    "width": "int",
    "height": "int",

    "motion_score": "float",
    "scene_change_score": "float",

    "created_by_notebook": "str",
}

EVIDENCE_COLUMNS = list(EVIDENCE_SCHEMA.keys())

# ------------------------------------------------------------
# Required Columns for Validation
# ------------------------------------------------------------

REQUIRED_EVIDENCE_COLUMNS = [
    "evidence_id",
    "video_id",
    "split",
    "source_video_path",
    "start_time_sec",
    "midpoint_time_sec",
    "end_time_sec",
    "duration_sec",
    "start_frame_idx",
    "midpoint_frame_idx",
    "end_frame_idx",
]

# ------------------------------------------------------------
# Columns Expected to Contain Unique Values
# ------------------------------------------------------------

UNIQUE_EVIDENCE_COLUMNS = [
    "evidence_id",
]

# ------------------------------------------------------------
# Columns Used for Downstream Retrieval
# ------------------------------------------------------------

RETRIEVAL_REFERENCE_COLUMNS = [
    "evidence_id",
    "video_id",
    "source_video_path",
    "start_time_sec",
    "midpoint_time_sec",
    "end_time_sec",
    "start_frame_idx",
    "midpoint_frame_idx",
    "end_frame_idx",
]

# ------------------------------------------------------------
# Display Schema Summary
# ------------------------------------------------------------

print("Evidence metadata schema defined successfully.")
print(f"Schema columns: {len(EVIDENCE_COLUMNS)}")
print(f"Required columns: {len(REQUIRED_EVIDENCE_COLUMNS)}")
print(f"Unique columns: {len(UNIQUE_EVIDENCE_COLUMNS)}")

if VERBOSE:

    print("\nEvidence Metadata Columns")
    print("-" * 60)

    for column_name, data_type in EVIDENCE_SCHEMA.items():
        print(f"{column_name:<24} {data_type}")



#### Evidence Metadata Field Definitions

| Field | Description |
|---------|-------------|
| `evidence_id` | Unique identifier assigned to each evidence unit. |
| `video_id` | NExT-QA video identifier associated with the evidence unit. |
| `split` | Dataset split associated with the source video or related QA records (`train`, `val`, or `test`). |
| `source_video_path` | Local path to the source video file used to generate the evidence unit. |
| `evidence_level` | Hierarchy level of the evidence unit. Level `0` typically represents a parent or top-level segment. |
| `parent_evidence_id` | Identifier of the parent evidence unit when hierarchical segmentation is used. Empty for top-level evidence units. |
| `segment_strategy` | Segmentation method used to create the evidence unit, such as fixed-duration, scene-based, or motion-based segmentation. |
| `start_time_sec` | Segment start time in seconds from the beginning of the source video. |
| `midpoint_time_sec` | Segment midpoint time in seconds, used as a representative temporal reference. |
| `end_time_sec` | Segment end time in seconds from the beginning of the source video. |
| `duration_sec` | Duration of the evidence segment in seconds. |
| `start_frame_idx` | Frame index corresponding to the segment start time. |
| `midpoint_frame_idx` | Frame index corresponding to the segment midpoint time. |
| `end_frame_idx` | Frame index corresponding to the segment end time. |
| `fps` | Frames per second of the source video. |
| `frame_count` | Total number of frames in the source video. |
| `width` | Source video frame width in pixels. |
| `height` | Source video frame height in pixels. |
| `motion_score` | Numeric estimate of motion or visual activity within the evidence segment. |
| `scene_change_score` | Numeric estimate of scene-transition strength associated with the evidence segment. |
| `created_by_notebook` | Notebook identifier used to record which notebook generated the evidence metadata. |

### 🔷 Step 7 — Define Evidence Segmentation Parameters

* Define the parameters used to partition videos into evidence segments.
* Specify segment duration limits, sampling intervals, and boundary selection criteria.
* Configure start, midpoint, and end timestamp generation for each segment.
* Define parent-child relationships for hierarchical evidence segmentation.
* Establish segmentation settings used throughout evidence generation.



In [ ]:
# ============================================================
# Step 7: Define Evidence Segmentation Parameters
# ============================================================

# ------------------------------------------------------------
# Segmentation Strategy
# ------------------------------------------------------------

SEGMENT_STRATEGY = "fixed_duration"

# ------------------------------------------------------------
# Segment Duration Settings
# ------------------------------------------------------------

MIN_SEGMENT_DURATION_SEC = 4.0

MAX_SEGMENT_DURATION_SEC = 8.0

DEFAULT_SEGMENT_DURATION_SEC = 6.0

# ------------------------------------------------------------
# Frame Reference Settings
# ------------------------------------------------------------

INCLUDE_START_FRAME = True

INCLUDE_MIDPOINT_FRAME = True

INCLUDE_END_FRAME = True

# ------------------------------------------------------------
# Evidence Hierarchy Settings
# ------------------------------------------------------------

ENABLE_PARENT_EVIDENCE = False

PARENT_SEGMENT_DURATION_SEC = None

EVIDENCE_LEVEL = 0

# ------------------------------------------------------------
# Motion and Scene Metrics
# ------------------------------------------------------------

COMPUTE_MOTION_SCORE = False

COMPUTE_SCENE_CHANGE_SCORE = False

DEFAULT_MOTION_SCORE = 0.0

DEFAULT_SCENE_CHANGE_SCORE = 0.0

# ------------------------------------------------------------
# Processing Limits
# ------------------------------------------------------------

MAX_VIDEOS_TO_PROCESS = None

SAMPLE_VIDEO_COUNT = 5

# ------------------------------------------------------------
# Validate Parameter Settings
# ------------------------------------------------------------

if MIN_SEGMENT_DURATION_SEC <= 0:
    raise ValueError("MIN_SEGMENT_DURATION_SEC must be greater than zero.")

if MAX_SEGMENT_DURATION_SEC < MIN_SEGMENT_DURATION_SEC:
    raise ValueError(
        "MAX_SEGMENT_DURATION_SEC must be greater than or equal to "
        "MIN_SEGMENT_DURATION_SEC."
    )

if not (
    MIN_SEGMENT_DURATION_SEC
    <= DEFAULT_SEGMENT_DURATION_SEC
    <= MAX_SEGMENT_DURATION_SEC
):
    raise ValueError(
        "DEFAULT_SEGMENT_DURATION_SEC must be between "
        "MIN_SEGMENT_DURATION_SEC and MAX_SEGMENT_DURATION_SEC."
    )

if SAMPLE_VIDEO_COUNT <= 0:
    raise ValueError("SAMPLE_VIDEO_COUNT must be greater than zero.")

# ------------------------------------------------------------
# Assemble Parameter Summary
# ------------------------------------------------------------

EVIDENCE_SEGMENTATION_PARAMETERS = {
    "segment_strategy": SEGMENT_STRATEGY,
    "min_segment_duration_sec": MIN_SEGMENT_DURATION_SEC,
    "max_segment_duration_sec": MAX_SEGMENT_DURATION_SEC,
    "default_segment_duration_sec": DEFAULT_SEGMENT_DURATION_SEC,
    "include_start_frame": INCLUDE_START_FRAME,
    "include_midpoint_frame": INCLUDE_MIDPOINT_FRAME,
    "include_end_frame": INCLUDE_END_FRAME,
    "enable_parent_evidence": ENABLE_PARENT_EVIDENCE,
    "parent_segment_duration_sec": PARENT_SEGMENT_DURATION_SEC,
    "evidence_level": EVIDENCE_LEVEL,
    "compute_motion_score": COMPUTE_MOTION_SCORE,
    "compute_scene_change_score": COMPUTE_SCENE_CHANGE_SCORE,
    "default_motion_score": DEFAULT_MOTION_SCORE,
    "default_scene_change_score": DEFAULT_SCENE_CHANGE_SCORE,
    "max_videos_to_process": MAX_VIDEOS_TO_PROCESS,
    "sample_video_count": SAMPLE_VIDEO_COUNT,
}

# ------------------------------------------------------------
# Display Summary
# ------------------------------------------------------------

print("Evidence segmentation parameters defined successfully.")

print(f"Segment strategy : {SEGMENT_STRATEGY}")
print(f"Default duration : {DEFAULT_SEGMENT_DURATION_SEC:.1f} seconds")
print(f"Duration range   : {MIN_SEGMENT_DURATION_SEC:.1f}–{MAX_SEGMENT_DURATION_SEC:.1f} seconds")
print(f"Parent evidence  : {ENABLE_PARENT_EVIDENCE}")

if VERBOSE:

    print("\nEvidence Segmentation Parameters")
    print("-" * 60)

    for parameter_name, parameter_value in EVIDENCE_SEGMENTATION_PARAMETERS.items():
        print(f"{parameter_name:<32} {parameter_value}")



### 🔷 Step 8 — Inspect Sample Videos

* Select representative videos from the NExT-QA dataset for inspection.
* Extract basic video properties including duration, frame count, frame rate, and resolution.
* Verify that video files can be successfully opened and processed.
* Review video characteristics relevant to evidence generation.
* Generate summary statistics for the inspected videos.



In [ ]:
# ============================================================
# Step 8: Inspect Sample Videos
# ============================================================

# ------------------------------------------------------------
# Third-Party Video Library Imports
# ------------------------------------------------------------

import cv2

# ------------------------------------------------------------
# Select Sample Videos
# ------------------------------------------------------------

sample_video_inventory_df = (
    video_inventory_df
    .sample(
        n=min(SAMPLE_VIDEO_COUNT, len(video_inventory_df)),
        random_state=42,
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Inspect Video Properties
# ------------------------------------------------------------

sample_video_records = []

for _, row in sample_video_inventory_df.iterrows():

    video_path = Path(row["video_path"])

    capture = cv2.VideoCapture(str(video_path))

    if not capture.isOpened():

        sample_video_records.append(
            {
                "video_id": row["video_id"],
                "video_path": str(video_path),
                "readable": False,
                "fps": None,
                "frame_count": None,
                "duration_sec": None,
                "width": None,
                "height": None,
            }
        )

        continue

    fps = capture.get(cv2.CAP_PROP_FPS)
    frame_count = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))

    duration_sec = (
        frame_count / fps
        if fps and fps > 0
        else None
    )

    capture.release()

    sample_video_records.append(
        {
            "video_id": row["video_id"],
            "video_path": str(video_path),
            "readable": True,
            "fps": fps,
            "frame_count": frame_count,
            "duration_sec": duration_sec,
            "width": width,
            "height": height,
        }
    )

sample_video_properties_df = pd.DataFrame(sample_video_records)

# ------------------------------------------------------------
# Display Inspection Results
# ------------------------------------------------------------

readable_count = sample_video_properties_df["readable"].sum()

print("Sample video inspection complete.")
print(f"Sample videos inspected : {len(sample_video_properties_df)}")
print(f"Readable videos         : {readable_count}")

if VERBOSE:

    print("\nSample Video Properties")
    print("-" * 60)

    display(sample_video_properties_df)



### 🔷 Step 9 — Generate Evidence Metadata Records

* Process NExT-QA videos using the defined evidence segmentation parameters.
* Generate evidence records containing video identifiers, timestamps, and segment metadata.
* Assign unique identifiers and maintain links to source videos.
* Compute evidence attributes required for validation and downstream processing.
* Assemble evidence records into a structured dataset.


In [ ]:
# ============================================================
# Step 9: Generate Evidence Metadata Records
# ============================================================

# ------------------------------------------------------------
# Helper: Determine Split Assignment by Video
# ------------------------------------------------------------

video_split_lookup = (
    annotations_df
    .groupby("video_id")["split"]
    .apply(lambda values: ",".join(sorted(set(values.dropna()))))
    .to_dict()
)

# ------------------------------------------------------------
# Select Videos for Evidence Generation
# ------------------------------------------------------------

videos_to_process_df = video_inventory_df.copy()

if MAX_VIDEOS_TO_PROCESS is not None:

    videos_to_process_df = (
        videos_to_process_df
        .head(MAX_VIDEOS_TO_PROCESS)
        .reset_index(drop=True)
    )

print("Generating evidence metadata records...")
print(f"Videos selected for processing: {len(videos_to_process_df):,}")

# ------------------------------------------------------------
# Generate Fixed-Duration Evidence Records
# ------------------------------------------------------------

evidence_records = []
failed_video_records = []

for video_number, (_, video_row) in enumerate(
    videos_to_process_df.iterrows(),
    start=1,
):

    video_id = str(video_row["video_id"])
    source_video_path = Path(video_row["video_path"])

    capture = cv2.VideoCapture(str(source_video_path))

    if not capture.isOpened():

        failed_video_records.append(
            {
                "video_id": video_id,
                "source_video_path": str(source_video_path),
                "error": "Unable to open video file",
            }
        )

        continue

    fps = float(capture.get(cv2.CAP_PROP_FPS))
    frame_count = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))

    capture.release()

    if fps <= 0 or frame_count <= 0:

        failed_video_records.append(
            {
                "video_id": video_id,
                "source_video_path": str(source_video_path),
                "error": "Invalid FPS or frame count",
            }
        )

        continue

    video_duration_sec = frame_count / fps

    segment_boundaries = []
    current_start_sec = 0.0

    while current_start_sec < video_duration_sec:

        current_end_sec = min(
            current_start_sec + DEFAULT_SEGMENT_DURATION_SEC,
            video_duration_sec,
        )

        current_duration_sec = current_end_sec - current_start_sec

        if (
            current_duration_sec < MIN_SEGMENT_DURATION_SEC
            and segment_boundaries
        ):

            previous_start_sec, _ = segment_boundaries[-1]

            segment_boundaries[-1] = (
                previous_start_sec,
                current_end_sec,
            )

        else:

            segment_boundaries.append(
                (
                    current_start_sec,
                    current_end_sec,
                )
            )

        current_start_sec = current_end_sec

    for segment_index, (start_time_sec, end_time_sec) in enumerate(
        segment_boundaries,
        start=1,
    ):

        midpoint_time_sec = (
            start_time_sec + end_time_sec
        ) / 2.0

        duration_sec = end_time_sec - start_time_sec

        start_frame_idx = max(
            0,
            min(
                frame_count - 1,
                int(round(start_time_sec * fps)),
            ),
        )

        midpoint_frame_idx = max(
            0,
            min(
                frame_count - 1,
                int(round(midpoint_time_sec * fps)),
            ),
        )

        end_frame_idx = max(
            0,
            min(
                frame_count - 1,
                int(round(end_time_sec * fps)) - 1,
            ),
        )

        evidence_id = f"{video_id}__ev_{segment_index:04d}"

        evidence_records.append(
            {
                "evidence_id": evidence_id,
                "video_id": video_id,
                "split": video_split_lookup.get(video_id, ""),
                "video_path": str(source_video_path),
                "segment_index": segment_index,
                "evidence_level": EVIDENCE_LEVEL,
                "parent_evidence_id": "",
                "segment_strategy": SEGMENT_STRATEGY,
                "start_time_sec": round(start_time_sec, 3),
                "midpoint_time_sec": round(midpoint_time_sec, 3),
                "end_time_sec": round(end_time_sec, 3),
                "duration_sec": round(duration_sec, 3),
                "start_frame_idx": start_frame_idx,
                "midpoint_frame_idx": midpoint_frame_idx,
                "end_frame_idx": end_frame_idx,
                "fps": round(fps, 3),
                "frame_count": frame_count,
                "width": width,
                "height": height,
                "motion_score": DEFAULT_MOTION_SCORE,
                "scene_change_score": DEFAULT_SCENE_CHANGE_SCORE,
                "created_by_notebook": "02_Prepare_Video_Evidence",
            }
        )

    if VERBOSE and video_number % 500 == 0:

        print(
            f"  Processed {video_number:,} / "
            f"{len(videos_to_process_df):,} videos"
        )

# ------------------------------------------------------------
# Assemble Evidence Metadata DataFrame
# ------------------------------------------------------------

evidence_metadata_df = pd.DataFrame.from_records(
    evidence_records,
    columns=EVIDENCE_COLUMNS,
)

failed_videos_df = pd.DataFrame.from_records(
    failed_video_records,
)

# ------------------------------------------------------------
# Display Summary
# ------------------------------------------------------------

print("\nEvidence metadata generation complete.")
print(f"Evidence records generated : {len(evidence_metadata_df):,}")
print(f"Videos processed           : {len(videos_to_process_df):,}")
print(f"Failed videos              : {len(failed_videos_df):,}")

if VERBOSE:

    print("\nEvidence Metadata Sample")
    print("-" * 60)
    display(evidence_metadata_df.head())

    if not failed_videos_df.empty:

        print("\nFailed Video Records")
        print("-" * 60)
        display(failed_videos_df)



### 🔷 Step 10 — Validate Evidence Metadata

* Verify that generated evidence records conform to the defined metadata schema.
* Validate required fields, data types, timestamps, and evidence relationships.
* Confirm that evidence records reference valid source videos.
* Identify missing, duplicate, or inconsistent metadata entries.
* Generate validation statistics and quality metrics for the evidence dataset.



In [ ]:
# ============================================================
# Step 10: Validate Evidence Metadata
# ============================================================

# ------------------------------------------------------------
# Run Validation
# ------------------------------------------------------------

validation_summary = validate_evidence_metadata(
    evidence_metadata=evidence_metadata_df,
    verbose=VERBOSE,
)

# ------------------------------------------------------------
# Convert Validation Issues to DataFrame
# ------------------------------------------------------------

validation_issues_df = (
    validation_issues_to_dataframe(
        validation_summary
    )
)

# ------------------------------------------------------------
# Display Summary
# ------------------------------------------------------------

print("\nEvidence metadata validation complete.")

print(
    f"Evidence records validated : "
    f"{validation_summary['record_count']:,}"
)

print(
    f"Errors                     : "
    f"{validation_summary['error_count']}"
)

print(
    f"Warnings                   : "
    f"{validation_summary['warning_count']}"
)

print(
    f"Validation Passed          : "
    f"{validation_summary['passed']}"
)

if VERBOSE and not validation_issues_df.empty:

    print("\nValidation Issues")
    print("-" * 60)

    display(validation_issues_df)



### 🔷 Step 11 — Save Evidence Metadata and Summary Files

* Save the validated evidence metadata dataset to the project output directories.
* Generate summary files describing evidence records and video coverage.
* Export metadata files required for downstream processing and analysis.
* Preserve evidence schema information and processing statistics.
* Verify that all output files were successfully written.



In [ ]:
# ============================================================
# Step 11: Save Evidence Metadata and Summary Files
# ============================================================

# ------------------------------------------------------------
# Build Evidence Summary
# ------------------------------------------------------------

unique_video_count = (
    evidence_metadata_df["video_id"]
    .nunique()
)

average_evidence_per_video = (
    len(evidence_metadata_df)
    / unique_video_count
)

average_segment_duration_sec = (
    evidence_metadata_df["duration_sec"]
    .mean()
)

evidence_summary_records = [
    {
        "metric": "evidence_record_count",
        "value": len(evidence_metadata_df),
    },
    {
        "metric": "unique_video_count",
        "value": unique_video_count,
    },
    {
        "metric": "average_evidence_per_video",
        "value": round(
            average_evidence_per_video,
            2,
        ),
    },
    {
        "metric": "average_segment_duration_sec",
        "value": round(
            average_segment_duration_sec,
            3,
        ),
    },
    {
        "metric": "segment_strategy",
        "value": SEGMENT_STRATEGY,
    },
    {
        "metric": "default_segment_duration_sec",
        "value": DEFAULT_SEGMENT_DURATION_SEC,
    },
    {
        "metric": "validation_passed",
        "value": validation_summary["passed"],
    },
    {
        "metric": "validation_error_count",
        "value": validation_summary["error_count"],
    },
    {
        "metric": "validation_warning_count",
        "value": validation_summary["warning_count"],
    },
]

evidence_summary_df = pd.DataFrame.from_records(
    evidence_summary_records
)

# ------------------------------------------------------------
# Save Evidence Metadata and Summary Files
# ------------------------------------------------------------

evidence_metadata_df.to_csv(
    EVIDENCE_METADATA_CSV,
    index=False,
)

evidence_summary_df.to_csv(
    EVIDENCE_SUMMARY_CSV,
    index=False,
)

# ------------------------------------------------------------
# Save Validation Issues When Present
# ------------------------------------------------------------

EVIDENCE_VALIDATION_CSV = (
    EVIDENCE_REPORTS_DIR /
    "evidence_validation.csv"
)

if not validation_issues_df.empty:

    validation_issues_df.to_csv(
        EVIDENCE_VALIDATION_CSV,
        index=False,
    )

else:

    EVIDENCE_VALIDATION_CSV = None

# ------------------------------------------------------------
# Verify Output Files
# ------------------------------------------------------------

required_output_files = [
    EVIDENCE_METADATA_CSV,
    EVIDENCE_SUMMARY_CSV,
]

for output_file in required_output_files:

    if not output_file.exists():

        raise FileNotFoundError(
            f"Expected output file was not created: "
            f"{output_file}"
        )

# ------------------------------------------------------------
# Display Save Summary
# ------------------------------------------------------------

print(
    "Evidence metadata and summary files "
    "saved successfully."
)

print(
    f"Evidence metadata : "
    f"{EVIDENCE_METADATA_CSV}"
)

print(
    f"Evidence summary  : "
    f"{EVIDENCE_SUMMARY_CSV}"
)

if EVIDENCE_VALIDATION_CSV is not None:

    print(
        f"Validation issues : "
        f"{EVIDENCE_VALIDATION_CSV}"
    )

if VERBOSE:

    print("\nEvidence Summary")
    print("-" * 60)

    display(evidence_summary_df)

    print("\nSaved File Sizes")
    print("-" * 60)

    for output_file in required_output_files:

        file_size_mb = (
            output_file.stat().st_size
            / (1024 ** 2)
        )

        print(
            f"{output_file.name:<28} "
            f"{file_size_mb:>10.2f} MB"
        )



### 🔷 Step 12 — Preview Sample Evidence Units

* Display a representative sample of generated evidence metadata records.
* Review evidence identifiers, video references, timestamps, and segment relationships.
* Verify that evidence records accurately represent the intended video segments.
* Inspect summary statistics for the generated evidence dataset.
* Confirm that the evidence metadata is complete and ready for downstream processing.



In [ ]:
# ============================================================
# Step 12: Preview Sample Evidence Units
# ============================================================

# ------------------------------------------------------------
# Select Sample Evidence Records
# ------------------------------------------------------------

sample_evidence_df = (
    evidence_metadata_df
    .sample(
        n=min(SAMPLE_VIDEO_COUNT, len(evidence_metadata_df)),
        random_state=42,
    )
    .sort_values(
        by=[
            "video_id",
            "segment_index",
        ]
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Display Sample Evidence Records
# ------------------------------------------------------------

print("Sample evidence units selected.")

print(f"Sample evidence records : {len(sample_evidence_df)}")

if VERBOSE:

    print("\nSample Evidence Metadata")
    print("-" * 60)

    display(sample_evidence_df)

# ------------------------------------------------------------
# Display Evidence Coverage by Split
# ------------------------------------------------------------

evidence_split_summary_df = (
    evidence_metadata_df
    .groupby("split", dropna=False)
    .agg(
        evidence_count=("evidence_id", "count"),
        unique_video_count=("video_id", "nunique"),
        average_duration_sec=("duration_sec", "mean"),
    )
    .reset_index()
)

evidence_split_summary_df["average_duration_sec"] = (
    evidence_split_summary_df["average_duration_sec"]
    .round(3)
)

print("\nEvidence coverage by split:")

display(evidence_split_summary_df)

# ------------------------------------------------------------
# Display Evidence Duration Summary
# ------------------------------------------------------------

evidence_duration_summary_df = (
    evidence_metadata_df["duration_sec"]
    .describe()
    .to_frame(name="duration_sec")
)

print("\nEvidence duration summary:")

display(evidence_duration_summary_df)



### 🔷 Step 13 — Notebook Summary

* Review the evidence metadata generation process and resulting outputs.
* Summarize video coverage, evidence record counts, and validation results.
* Confirm that the evidence metadata dataset was successfully generated and saved.
* Verify readiness for downstream processing and analysis.
* Identify any issues or recommendations for subsequent notebooks.



In [ ]:
# ============================================================
# Step 13: Notebook Summary
# ============================================================

# ------------------------------------------------------------
# Summarize Notebook Outputs
# ------------------------------------------------------------

print("Notebook 02 complete.")
print("=" * 60)

print("\nPrimary Output")
print("-" * 60)
print(f"Evidence metadata CSV : {EVIDENCE_METADATA_CSV}")
print(f"Evidence summary CSV  : {EVIDENCE_SUMMARY_CSV}")

print("\nEvidence Generation Summary")
print("-" * 60)
print(f"Videos processed          : {evidence_metadata_df['video_id'].nunique():,}")
print(f"Evidence records created  : {len(evidence_metadata_df):,}")
print(f"Segmentation strategy     : {SEGMENT_STRATEGY}")
print(f"Default segment duration  : {DEFAULT_SEGMENT_DURATION_SEC:.1f} seconds")

print("\nValidation Summary")
print("-" * 60)
print(f"Validation passed         : {validation_summary['passed']}")
print(f"Validation errors         : {validation_summary['error_count']}")
print(f"Validation warnings       : {validation_summary['warning_count']}")

